# 03 Robustness checks

**Data:** chosen in `common.py` (`USE_FAKE_DATA`). Chart titles say whether it is real or fake.

Do the conclusions still hold when reasonable choices change?

1. Rising threshold: 1.1, 1.2, 1.3
2. Period: full test, 2025 only, without spike weeks
3. Top-K for buyers: 5, 10, 20 styles per week
4. Random seeds (only when the file has a `seed` column)

Each table ends with **beats best baseline?**: does the main model score higher than the best baseline in that setting?

In [ ]:
import pandas as pd

from common import load_predictions, load_seed_runs, spike_weeks, RISING_THRESHOLD, MAIN_MODEL, BASELINES

MODEL = MAIN_MODEL
T = RISING_THRESHOLD  # the team's chosen threshold
HORIZON = 4  # change to 8 to rerun every check for 8 weeks ahead

df = load_predictions()
df = df[(df["horizon"] == HORIZON) & df["model"].isin([MODEL] + BASELINES)].copy()

# Spike weeks flagged during data preparation (spike_weeks.csv)
SPIKE_WEEKS = spike_weeks()

# A forecast "touches the spike" if any of the 4 weeks it predicts is a spike week.
def touches_spike(row):
    window_end = row["origin"] + pd.Timedelta(weeks=int(row["horizon"]))
    window_start = window_end - pd.Timedelta(weeks=3)
    return any(window_start <= week <= window_end for week in SPIKE_WEEKS)

df["touches_spike"] = df.apply(touches_spike, axis=1)
print(f"{len(df):,} forecasts; {df['touches_spike'].mean():.0%} of them touch spike weeks")

## Shared scoring

Precision and recall as in notebook 01, for any threshold.

In [ ]:
def precision_recall(group, threshold):
    said = group["predicted_growth"] > threshold
    rose = group["actual_growth"] > threshold
    correct = (said & rose).sum()
    precision = 100 * correct / said.sum() if said.sum() else float("nan")
    recall = 100 * correct / rose.sum() if rose.sum() else float("nan")
    return round(precision, 1), round(recall, 1)


def compare(data, threshold):
    """One row: main model and best baseline, with the yes/no verdict."""
    p_main, r_main = precision_recall(data[data["model"] == MODEL], threshold)
    baseline_scores = {b: precision_recall(data[data["model"] == b], threshold) for b in BASELINES}
    best = max(baseline_scores, key=lambda b: baseline_scores[b][0] + baseline_scores[b][1])
    p_base, r_base = baseline_scores[best]
    return {
        f"{MODEL} precision": p_main, f"{MODEL} recall": r_main,
        "best baseline": best, "baseline precision": p_base, "baseline recall": r_base,
        "beats best baseline?": "yes" if (p_main >= p_base and r_main >= r_base) else "no",
    }

## 1. Different thresholds

"Beats" means at least as good on **both** precision and recall.

In [ ]:
rows = {f"threshold {t}": compare(df, t) for t in (1.1, 1.2, 1.3)}
pd.DataFrame(rows).T

## 2. Different periods (team threshold)

The proposal promised all three views, because the spring 2026 spike could make the test period look better or worse than normal.

In [ ]:
periods = {
    "full test period": df,
    "2025 only": df[df["origin"].dt.year == 2025],
    "without spike weeks": df[~df["touches_spike"]],
}
pd.DataFrame({name: compare(data, T) for name, data in periods.items()}).T

## 3. Top-K for buyers

Each week, take the K styles with the highest predicted growth. **Precision at K** = share of those K that actually rose (growth above the team threshold), averaged over all weeks.

This is how a buyer would use the app: stock up on the top K.

In [ ]:
def precision_at_k(data, k, threshold=T):
    weekly = []
    for _, week in data.groupby("origin"):
        top = week.nlargest(k, "predicted_growth")
        weekly.append((top["actual_growth"] > threshold).mean())
    return round(100 * sum(weekly) / len(weekly), 1)


rows = {}
for k in (5, 10, 20):
    scores = {m: precision_at_k(df[df["model"] == m], k) for m in [MODEL] + BASELINES}
    best_baseline = max(scores[b] for b in BASELINES)
    rows[f"top {k}"] = {**{f"{m} (%)": s for m, s in scores.items()},
                        "beats best baseline?": "yes" if scores[MODEL] >= best_baseline else "no"}

# For comparison: picking K styles at random would give roughly the share of styles that rise.
print(f"Random picks would be right about {100 * (df[df['model'] == MODEL]['actual_growth'] > T).mean():.1f}% of the time")
pd.DataFrame(rows).T

## 4. Random seeds

Tree models give slightly different results each time they are trained. If the file has a `seed` column (5 runs), this shows the spread of precision and recall across runs.

In [ ]:
full = load_seed_runs()
if "seed" not in full.columns:
    print("No `seed` column in this file yet: this check runs once the predictions file includes it.")
else:
    runs = full[full["horizon"] == HORIZON]
    per_seed = pd.DataFrame(
        {seed: dict(zip(["precision", "recall"], precision_recall(group, T))) for seed, group in runs.groupby(runs["seed"].astype(int))}
    ).T
    print(per_seed)
    print("Spread (max - min):", (per_seed.max() - per_seed.min()).round(1).to_dict())

## What this shows

Real data, 137 styles, 4 weeks ahead, test period January 2025 – September 2026.

1. **Thresholds:** over the full test period, LightGBM does not beat the best baseline at any of
   1.1, 1.2 or 1.3 (precision 55.5% / 39.6% / 25.7% vs. 69.0% / 61.1% / 57.3% for last value).
   Outside May–July 2026 its precision is higher at all three (79.3% / 76.1% / 78.2% vs.
   71.5% / 62.5% / 57.6%). The post-spike months decide the result, not the threshold.
2. **Periods:** in 2025 only, LightGBM beats every baseline (precision 68.8%, recall 39.5%; last
   value 49.0% / 9.7%). Without spike weeks it does not (32.0% / 35.0% vs. 39.9% / 12.9%), because
   85% of the May–July 2026 forecasts target weeks after the spike and stay in this view.
3. **Top-K:** a buyer stocking LightGBM's top 10 each week would be right 54.7% of the time, vs.
   42.9% for last value and 20.1% for random picks. LightGBM wins at top 5, 10 and 20.
4. **Seeds:** precision varies from 39.2% to 40.9% and recall from 27.8% to 29.5% across the
   5 runs (spread under 2 points), so the results do not depend on a lucky run.